**COMMAND:** `pgrep`

**PURPOSE (support):** You already do `ps aux | grep nginx` to find a PID — but that's fragile: it matches its own grep process, needs `awk` to extract the PID column, and breaks in scripts.

> `pgrep` returns just the PID(s), cleanly, so you can pipe it straight into `kill` or use it in a health-check script.

**SYNTAX:** `pgrep [flags] pattern`

**FLAGS:**

|Flag|Meaning|
|---|---|
|`-l`|show process name alongside PID|
|`-a`|show full command line alongside PID|
|`-u user`|only match processes owned by that user|
|`-f`|match against full command line, not just process name|
|`-x`|exact match only|
|`-c`|count matching processes|

**EXAMPLE:**

In [ ]:
$ pgrep -a bash
1423 -bash
2891 -bash

READ THE OUTPUT: Two PIDs, both bash shells — probably your current terminal plus another tab/pane. Compare to the noisy version:

In [ ]:
$ ps aux | grep bash
dennie   1423  0.0  0.1  ...  -bash
dennie   2891  0.0  0.1  ...  -bash
dennie   3355  0.0  0.0  ...  grep --color=auto bash

See that last line?
> `ps aux | grep bash` always matches its own grep call. `pgrep` never has that problem — it excludes itself.

In [ ]:
dennie@HP:~/linux-tut$ pgrep -a bash
375 -bash
604 /bin/bash --init-file /home/dennie/.vscode-server/bin/a44adf7f53e00964ab890f9f8758a334f1fc15bc/out/vs/workbench/contrib/terminal/common/scripts/shellIntegration-bash.sh
3743 /bin/bash --init-file /home/dennie/.vscode-server/bin/a44adf7f53e00964ab890f9f8758a334f1fc15bc/out/vs/workbench/contrib/terminal/common/scripts/shellIntegration-bash.sh

In [ ]:
dennie@HP:~/linux-tut$ pgrep -u dennie -l bash
457 bash
1556 bash

In [ ]:
dennie@HP:~/linux-tut$ pgrep -c ssh
0

Run each and check the output makes sense — do the PIDs match what you'd expect from `ps aux`?

**TICKET SCENARIO:** "The deployment script keeps starting a second instance of our worker process." You'd check with `pgrep -f worker.py` before launch — if it returns a PID, don't start a second one. This is the classic dev bridge: it's a lockfile check, just process-based instead of file-based.

**GOTCHA:** `pgrep somename` matches the process _name_ by default (max 15 chars on Linux, truncated), which can cause false negatives on long binary names. Use `-f` to match the full command line instead — e.g. `pgrep -f gunicorn` will catch `gunicorn: master [myapp]` even if `pgrep gunicorn` misses it due to name truncation.

---

**COMMAND:** `kill` (with signals) — the whole point of `kill` is which signal you send, not just "make it stop"

**PURPOSE (support):** As a Django dev, you've done `kill <pid>` or `kill -9 <pid>` without thinking much about it. In support, the signal you choose matters — SIGTERM asks a process to clean up and exit; SIGKILL yanks it out from under the OS with zero warning. Choosing wrong can corrupt data, leave lock files behind, or leave orphaned child processes.

**SYNTAX:** `kill -SIGNAL PID` or `kill -N PID` (numeric equivalent)

**FLAGS / SIGNALS:**

|Signal|Number|Meaning|Support use|
|---|---|---|---|
|`SIGTERM`|15|"Please terminate" — graceful, catchable|**Always try this first.** Gives the process a chance to close file handles, flush buffers, save state. This is what plain `kill PID` sends by default.|
|`SIGKILL`|9|"Terminate now" — not catchable, not ignorable, handled entirely by the kernel|Last resort — process is unresponsive/hung and SIGTERM didn't work. No cleanup happens. Can leave temp files, DB connections, or lock files dangling.|
|`SIGHUP`|1|Originally "terminal hung up"|Repurposed by most daemons to mean "reload your config without fully restarting" — e.g. `kill -HUP $(pgrep nginx-master-pid)` reloads nginx config in place|

**Developer bridge:** SIGTERM is like Django's `request_finished` signal firing during graceful shutdown — code gets to run cleanup logic. SIGKILL is like the OS pulling the power cord mid-`request` — nothing gets a chance to run. SIGHUP-as-reload is like sending Gunicorn a HUP to reload workers without dropping the listening socket — zero-downtime config reload, no new connections refused.

**EXAMPLE:**

In [ ]:
$ pgrep -a sleep
4821 sleep 300

$ kill -15 4821
$ ps aux | grep [s]leep
(no output — process exited cleanly)

Now the forceful version:

In [ ]:
$ sleep 300 &
[1] 5102

$ kill -9 5102
$ jobs
[1]+  Killed                  sleep 300

Note the shell reports "Killed" — that's the shell telling you the process was terminated by a signal it couldn't catch, distinct from the process exiting on its own.

**READ THE OUTPUT:** If `kill -15 PID` doesn't make the process disappear from `ps aux | grep` within a couple seconds, that's your signal to escalate to `kill -9 PID`. If SIGKILL _also_ doesn't remove it, you're not looking at a normal process anymore — you likely have a **zombie** (STAT `Z`, already dead, waiting on parent to reap it — killing it does nothing, you need to deal with the parent) or a process stuck in `D` state (uninterruptible I/O sleep — often a hardware/NFS/disk issue, SIGKILL literally cannot touch it).

In [ ]:
dennie@HP:~/linux-tut$ sleep 300 &
[1] 4223

Runs `sleep` (a command that does nothing but wait) for 300 seconds. The trailing `&` sends it to the **background** so you get your prompt back immediately instead of the terminal freezing for 5 minutes. The shell prints `[1] 4223` — `[1]` is the _job number_ (shell-local bookkeeping), `4223` is the _PID_ (kernel-wide, what you actually use with `kill`).

In [ ]:
dennie@HP:~/linux-tut$ pgrep -a sleep
4223 sleep 300

Asks the kernel "list any process whose name matches `sleep`," and `-a` prints the full command line next to each PID. Output: `4223 sleep 300` — confirms it's alive and tells you exactly what it was launched with.

In [ ]:
dennie@HP:~/linux-tut$ kill -15 $(pgrep sleep)
[1]+  Terminated              sleep 300

Two steps happen here: `$(pgrep sleep)` runs _first_, in a subshell, and returns just the bare PID (`4223`) since you didn't use `-a` this time. That output gets substituted into the outer command, so it becomes literally `kill -15 4223` — send SIGTERM to PID 4223. The shell then reports `[1]+ Terminated sleep 300`, confirming the process caught the signal and exited.

In [ ]:
dennie@HP:~/linux-tut$ pgrep -a sleep # no output, shown in red)

This is **not an error** — this is the correct, expected result. `pgrep` found zero matching processes (because you just killed the only one), so it printed nothing and exited with status code 1 instead of 0. Your terminal prompt (VS Code's shell integration) colors the next prompt red/highlighted whenever the previous command's exit code was non-zero — it's just telling you "that command's exit status wasn't success," not "something broke." For `pgrep`, "no matches" is a completely normal outcome and always exits non-zero. Don't be alarmed by that red flag specifically after `pgrep`.

Run again:

In [ ]:
dennie@HP:~/linux-tut$ sleep 300 &
[1] 4630

New sleep, new PID: `4630`.

In [ ]:
dennie@HP:~/linux-tut$ pgrep -a sleep
4630 sleep 300
4703 sleep 1

This one's interesting — it returned **two** lines.

`4630` is your sleep as expected. `4703 sleep 1` is _not_ something you ran — most likely VS Code's shell integration (the thing that draws those decorated prompts and tracks command status) silently runs a short-lived `sleep 1` in the background to help it detect when a command has finished. It's harmless and will exit on its own a second after you see it — nothing you need to kill deliberately, it won't outlive its own 1-second timer.

In [ ]:
dennie@HP:~/linux-tut$ kill - $(pgrep
 sleep)
bash: kill: : invalid signal specification

This is where it broke. You were mid-typing/pasting and it split across two lines, so bash ran `pgrep` with no arguments first (which alone would just hang waiting for... actually `pgrep` with no pattern errors out), and the outer command became `kill -` with nothing after the dash — bash read `-` as "you're about to give me a signal name" but found nothing there, hence `invalid signal specification`. The fix is just: never let `-15` and the PID get separated — type it as one clean line: `kill -15 $(pgrep sleep)`.

Now let's clean up. Check what's actually still running:

In [ ]:
dennie@HP:~/linux-tut$ pgrep -a sleep
dennie@HP:~/linux-tut$ pgrep -a sleep
7673 sleep 1
dennie@HP:~/linux-tut$ pgrep -a sleep
7718 sleep 1

Look at what changed between each run: the PID is different every time (`7673`, then `7718`), and it's always `sleep 1`, never `sleep 300`. That's the signature of a process being spawned fresh each time you press Enter, not a leftover hanging around. This is exactly the VS Code shell-integration probe I mentioned — every time it detects a new command, it kicks off a short `sleep 1` in the background to help sync prompt state, then it exits a second later. It's tied to VS Code's terminal, not to anything you launched.

Your actual test process (`sleep 300`, PID 4630) is gone — the earlier `kill -15 $(pgrep sleep)` did catch it, you just hadn't re-checked yet. No cleanup needed. **System is clean.**

One quick way to tell the two apart at a glance going forward: `sleep 1` = VS Code noise, ignore it. `sleep 300` (or whatever duration you actually typed) = yours.

Tell me what that prints — if `4630 sleep 300` is still listed (it likely is, since you never successfully killed it), run:

In [ ]:
kill -15 $(pgrep sleep)
pgrep -a sleep

and confirm it comes back empty (red prompt = expected here, not an error). Paste me the output and we'll confirm you're fully clean before moving to `nice`/`nohup`/job control.